In [ ]:
# ── Cell 1: Install ─────────────────────────────────────
!pip install ultralytics -q

In [ ]:
# ── Cell 2: Verify GPU ──────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
# ── Cell 3: Setup dataset.yaml ──────────────────────────
import shutil, yaml

shutil.copy(
    "/kaggle/input/datasets/singertv/fabric-processed/dataset.yaml",
    "/kaggle/working/dataset.yaml"
)

with open("/kaggle/working/dataset.yaml", "r") as f:
    data = yaml.safe_load(f)

data['path'] = "/kaggle/input/datasets/singertv/fabric-processed/processed"

with open("/kaggle/working/dataset.yaml", "w") as f:
    yaml.dump(data, f)

print("dataset.yaml ready ✅")
print(data)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data="/kaggle/working/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    device="0",
    project="/kaggle/working/fabric-defect",
    name="yolov8m-v1",
    patience=15,
    save=True,
    plots=True,
    workers=4,
    cache="disk",   
    amp=True
)

print("Training complete")



In [ ]:
# ── Cell 5: Validation ──────────────────────────────────
metrics = model.val()
print(f"\nmAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# ── Cell 6: Test set evaluation ─────────────────────────
test_results = model.val(
    data="/kaggle/working/dataset.yaml",
    split="test",
)

print("\n📊 Test Set Results:")
print(f"mAP50      : {test_results.box.map50:.4f}")
print(f"mAP50-95   : {test_results.box.map:.4f}")
print(f"Precision  : {test_results.box.mp:.4f}")
print(f"Recall     : {test_results.box.mr:.4f}")

# Per-class results
print("\n📊 Per-Class Results:")
names = ['Stain', 'Thread', 'Warp_Weft', 'hole', 'seam']
for i, name in enumerate(names):
    print(f"  {name:12s} — AP50: {test_results.box.ap50[i]:.4f}")

In [ ]:
# ── Cell 7: Save best.pt path ───────────────────────────
import os, shutil

best_pt = "/kaggle/working/fabric-defect/yolov8m-v1/weights/best.pt"
print(f"✅ best.pt exists: {os.path.exists(best_pt)}")

# Output tab এ directly দেখানোর জন্য copy করো
shutil.copy(best_pt, "/kaggle/working/best.pt")
print("⬇️  Download best.pt from Output tab")